> **How to use this notebook.** Step 5 of 5: the step-by-step audit. Needs: 01 (software); for steps (b) and (d) the inputs of 02 (`DATA_DIR`, `DERIVED_DIR`). Steps (a) and (c) need only the repository. Each step prints its result and records PASS / FAIL / NOT YET AVAILABLE; the last cell lists them.
>
> **Nurnet files (included).** The site list (`nurnet_nuraghi.shp`, 7,167 nuraghi with tower counts) and the per-site building stone file are Nurnet's data, included in `data/nurnet/`; 02 copies them to the names the configuration expects (`DATA_DIR/sites/`, `DATA_DIR/stone/`) and checks their SHA-256. For the latest version and source information contact Nurnet at nurnet.presidente@gmail.com. Details: [HOW_TO_USE.md](HOW_TO_USE.md#nurnet-files).
>
> Order: 01_setup, 02_inputs, 03_scenarios, 04_analysis_report, 05_audit (Jupyter or Google Colab). Full guide: [notebooks/HOW_TO_USE.md](HOW_TO_USE.md).

**Getting the data** (full table: `02_inputs.ipynb`, section 0; step by step: [HOW_TO_USE.md](HOW_TO_USE.md)).
Nurnet's site list and per-site building stone are included in `data/nurnet/` (for the latest version and source
information contact Nurnet at nurnet.presidente@gmail.com). Every other raw input is public: ARPAS lithology,
Copernicus DEM GLO-30 (AWS tiles), AGRIS soils (WFS), RAS rivers; `notebooks/fetch_data.py` downloads them from the
official sources and rebuilds the model files, or link them by hand (HOW_TO_USE.md, 'Linking the data yourself').
Derived inputs: the prebuilt pack (`DERIVED_URL`, when public) or `python notebooks/fetch_data.py --rebuild-derived`.

# 05 Audit: code, inputs, configuration, a rerun, and the headline numbers

| step | what is checked | against | expected result |
|---|---|---|---|
| (a) | every file of `code/` | `CODE_SHA256SUMS` (checksums of the run's frozen snapshot) and `CODE_PROVENANCE.md` | every file byte-identical to the run, or scrubbed and listed in CODE_PROVENANCE.md (comments / docstrings / paths only), or excluded and listed there; **0 unexplained differences** |
| (b) | every raw and derived input | `notebooks/inputs_manifest.csv` (SHA-256 of the copies used for the runs) | Nurnet files and soils **identical**; rivers, lithology, DEM, cost and slope *rebuilt (not byte-comparable)* when fetched from the official sources (identical when you use the run's copies); all 68 derived inputs present |
| (c) | every audited parameter | `param_audit/PARAM_TRACE.csv` (value of each parameter in `config_full_audited.yaml` after the audit) | **68 of 68** config values equal (2 rows are code-level, not config keys) |
| (d) | a fresh rerun of the headline scenario `MOST_REALISTIC_AUDITED_LIT_P10` on 300 sites (`code/sites300_real.txt`), seeds 1-3 | the shipped fingerprint `outputs/reference/reference_audit_sha256.csv` | **exact match** of every output item (placeholder: the fingerprint is generated after the full run ends; until then this step reports NOT YET AVAILABLE) |
| (e) | island years and island man-hours of the headline scenario | the shipped per-seed and summary tables `outputs/summary/per_seed_full_audited.csv`, `summary_full_audited.csv` | recomputed values equal to the shipped ones (placeholder until the results are public) |

Runtime: (a)-(c) under a minute; (b) with SHA-256 of ~1 GB of inputs about a minute; (d) a few minutes on 2 ranks
(alliance runs hold the whole island per rank: ~1-3 GB per rank); (e) seconds.

## Config (the only cell you edit)

All paths are set at the top of this cell. With `"auto"` the notebook uses the repository layout: `code/`,
`config/`, `data/` (raw inputs, see `data/README.md`), `data/derived/cache/` (derived inputs, see
`data/derived/README.md`) and `outputs/` (everything the notebooks write). Environment variables `NB_REPO_DIR`,
`NB_DATA_DIR`, `NB_DERIVED_DIR`, `NB_CONFIG`, `NB_OUTPUTS_DIR`, `NB_WORK_DIR`, `NB_RESULTS_DIR`, `NB_LOCK` override
the defaults (for headless runs with `jupyter nbconvert --execute`).

**Google Colab:** the repository is cloned to `/content/Repast_Nuraghe` by the setup cell. Each notebook runs in its own Colab runtime, so put the data and outputs on Google Drive: `MOUNT_DRIVE = True`, `DATA_DIR = "/content/drive/MyDrive/Repast_Nuraghe/data"`, `DERIVED_DIR = "/content/drive/MyDrive/Repast_Nuraghe/data/derived/cache"`, `OUTPUTS_DIR = "/content/drive/MyDrive/Repast_Nuraghe/outputs"` (the same values in every notebook).

`RUN_SUBSET` (step d): `"auto"` = rerun only when the reference fingerprint is present; `True` = always; `False` = never. `MAKE_AUDIT_REFERENCE = True` writes the fingerprint from this machine instead (maintainers, after the full run). `PARTS_DIR` (step e, optional): the part files of the headline scenario from the results package (`.../runs/alliance_MOST_REALISTIC_AUDITED_LIT_P10/<signature>/`), to recompute the per-seed table too.

In [ ]:
import os

# Repository layout: the only lines you may need to change. "auto" = the default folder of the repository.
REPO_DIR = os.environ.get("NB_REPO_DIR", "auto")         # the cloned repository ("auto": the parent of notebooks/)
REPO_URL = "https://github.com/maltaweel/Repast_Nuraghe"  # Colab: cloned to /content/Repast_Nuraghe if not found
DATA_DIR = os.environ.get("NB_DATA_DIR", "auto")         # raw inputs = data_dir ("auto": REPO_DIR/data; see data/README.md)
DERIVED_DIR = os.environ.get("NB_DERIVED_DIR", "auto")   # cached derived inputs ("auto": REPO_DIR/data/derived/cache)
CONFIG = os.environ.get("NB_CONFIG", "auto")             # base config ("auto": REPO_DIR/config/config_full_audited.yaml)
OUTPUTS_DIR = os.environ.get("NB_OUTPUTS_DIR", "auto")   # everything written here ("auto": REPO_DIR/outputs)
WORK_DIR = os.environ.get("NB_WORK_DIR", "auto")         # fast local disk for code + cache copies ("auto": OUTPUTS_DIR/work)
RESULTS_DIR = os.environ.get("NB_RESULTS_DIR", "auto")   # where runs are kept ("auto": OUTPUTS_DIR/notebook_runs)
MOUNT_DRIVE = False                                      # Colab only: mount Google Drive at /content/drive
INSTALL = True                                           # install OpenMPI + pinned packages if this runtime lacks them
LOCK = os.environ.get("NB_LOCK") or None                 # optional flock file (shared servers only)

RUN_SUBSET = os.environ.get("NB_RUN_SUBSET", "auto")    # step d: "auto" (if the reference exists) | True | False
MAKE_AUDIT_REFERENCE = os.environ.get("NB_MAKE_AUDIT_REFERENCE", "") == "1"   # step d: write the fingerprint instead
RANKS = min(2, os.cpu_count() or 1)                      # MPI ranks for step d
PARTS_DIR = os.environ.get("NB_PARTS_DIR") or None       # step e (optional): part files of the headline scenario


## Setup
Same cell in every notebook: repository (cloned on Colab), folders, install check.

In [ ]:
# Setup (no edits needed): find the repository, set the folders, install if needed, import the helpers in repro.py.
import os, sys, glob
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and MOUNT_DRIVE and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

def _is_repo(d):
    return os.path.isfile(os.path.join(d, "notebooks", "repro.py")) and os.path.isdir(os.path.join(d, "code"))

def _find_repo(p):
    cands = [p] if p != "auto" else [os.path.dirname(os.getcwd()), os.getcwd(), "/content/Repast_Nuraghe"]
    for c in cands:
        if _is_repo(c):
            return os.path.abspath(c)
    if IN_COLAB and p == "auto":                     # Colab opens only the notebook: clone the repository once
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/Repast_Nuraghe"], check=True)
        return "/content/Repast_Nuraghe"
    raise FileNotFoundError("Repository not found (tried " + ", ".join(cands) + "). Set REPO_DIR in the config cell.")

REPO_DIR = _find_repo(REPO_DIR)
PACKAGE_DIR = REPO_DIR                                     # name used by the helper calls below
sys.path.insert(0, os.path.join(REPO_DIR, "notebooks"))
import repro
if OUTPUTS_DIR == "auto":
    OUTPUTS_DIR = os.path.join(REPO_DIR, "outputs")
OUTPUTS_DIR = os.path.abspath(OUTPUTS_DIR)
L = repro.set_layout(repo=REPO_DIR, data=DATA_DIR, cache=DERIVED_DIR, config=CONFIG,
                     reference=os.path.join(OUTPUTS_DIR, "reference"))
DATA_DIR, DERIVED_DIR, CONFIG = L["data"], L["cache"], L["config"]
WORK_DIR = os.path.abspath(os.path.join(OUTPUTS_DIR, "work") if WORK_DIR == "auto" else WORK_DIR)
RESULTS_DIR = os.path.abspath(os.path.join(OUTPUTS_DIR, "notebook_runs") if RESULTS_DIR == "auto" else RESULTS_DIR)
for k in ("REPO_DIR", "DATA_DIR", "DERIVED_DIR", "CONFIG", "WORK_DIR", "RESULTS_DIR"):
    print(f"{k:12s}", globals()[k])
status = repro.ensure_env(install=INSTALL)
print("MPI:", repro.mpi_report(), "| Python", sys.version.split()[0], "| CPUs", os.cpu_count())
assert status == "ok", "Restart the kernel now (see message above), then run this notebook again."
import json, pandas as pd
import fetch_data
AUDIT = {}                                                # step -> result
REF_DIR = os.path.join(REPO_DIR, "outputs", "reference")  # shipped with the repository (fingerprint)
SUMMARY_DIR = os.path.join(REPO_DIR, "outputs", "summary")  # shipped with the repository (per-seed tables)

## (a) Code: checksums of the run's frozen snapshot
`CODE_SHA256SUMS` holds the SHA-256 of every `.py`, `.sh` and `.yaml` file of the code snapshot that ran the full
audited queue, written when it was frozen. `code/` is that snapshot scrubbed for publication (comments, docstrings,
paths; no model logic, parameter value or random draw: `CODE_PROVENANCE.md`). The cell runs the equivalent of
`cd code && sha256sum -c ../CODE_SHA256SUMS` and explains every result from `CODE_PROVENANCE.md`; it also checks that
`code/` is the published version (`inputs_manifest.csv`, group `code`).

**Expected result:** a table `OK / byte-identical to the run`, `FAILED / scrubbed (CODE_PROVENANCE.md)`, `missing /
excluded (CODE_PROVENANCE.md)` and `unexplained differences: 0`; then `code/ = published version: True`.

In [ ]:
code_table, code_bad = repro.code_audit(REPO_DIR)
try:
    repro._check_code(PACKAGE_DIR); same_pub = True
except repro.PreflightError as e:
    print(e); same_pub = False
print("code/ = published version (inputs_manifest.csv, group code):", same_pub)
AUDIT["a code"] = "PASS" if (len(code_bad) == 0 and same_pub) else "FAIL"
display(code_bad) if len(code_bad) else None
print("(a)", AUDIT["a code"])

## (b) Inputs: raw and derived inputs against the manifest
Raw inputs: SHA-256 of every file under `DATA_DIR` against `inputs_manifest.csv` (`fetch_data.py --check-only`; it
also copies the Nurnet files from `data/nurnet/` if they are not yet in place). Files that `fetch_data.py` rebuilt from
an official source are reported as *rebuilt (not byte-comparable)* with the reason (`DATA_DIR/fetch_provenance.json`).
Derived inputs: presence of the 68 cached derived inputs under `DERIVED_DIR`.

**Expected result:** `0 failed/differing` for the raw inputs (Nurnet files and soils *identical*; rivers,
lithology, DEM, cost and slope *identical* with the run's copies or *rebuilt (not byte-comparable)* from the official
sources) and `derived inputs in ...: 68 of 68 present`.

In [ ]:
raw = fetch_data.fetch(DATA_DIR, check_only=True)
prov = repro.fetch_provenance()
for r in raw:                                   # a file rebuilt by fetch_data.py: its recorded status
    p = prov.get(r["path"]) or {}
    if r["status"] == "differs" and str(p.get("status", "")).startswith("rebuilt") and p.get("sha256") == r["sha256"]:
        r["status"], r["note"] = p["status"], p.get("note", "")
raw_t = pd.DataFrame(raw)[["path", "status", "note"]]
display(raw_t)
derived_missing = fetch_data.derived_status(DERIVED_DIR)
bad_raw = raw_t[raw_t.status.isin(["differs", "FAILED", "missing"])]
AUDIT["b inputs"] = ("PASS" if bad_raw.empty and not derived_missing else
                     "FAIL" if not bad_raw.empty else "INCOMPLETE (derived inputs missing: 02_inputs.ipynb, section 0b)")
print("(b)", AUDIT["b inputs"])

## (c) Configuration: `config_full_audited.yaml` against `param_audit/PARAM_TRACE.csv`
`PARAM_TRACE.csv` lists the 70 audited parameters: config path, the value in `config_full_audited.yaml`, the audited
final value and where the frozen code reads it. `MR:<key>` paths are read from the resolved headline scenario
(`alliance.scenario(cfg, "MOST_REALISTIC_AUDITED_LIT_P10")` of the frozen code), the others from the YAML; each value is
compared with the trace as JSON text.

**Expected result:** `70 audited parameters; 68 config keys checked: 68 equal, 0 differ; 2 code-level rows (not config
keys); PARAM_TRACE 'value_matches_audit' = yes for 70`.

In [ ]:
cfg_t = repro.config_audit(REPO_DIR, CONFIG)
diff = cfg_t[cfg_t.same == False]                      # noqa: E712
display(diff if len(diff) else cfg_t.head(10))
AUDIT["c config"] = "PASS" if len(diff) == 0 and (cfg_t["audited = config (trace)"] == "yes").all() else "FAIL"
print("(c)", AUDIT["c config"])

## (d) Rerun a fixed subset and compare with the shipped fingerprint
The headline scenario `MOST_REALISTIC_AUDITED_LIT_P10` (config `config_full_audited.yaml`) on the 300 sites of
`code/sites300_real.txt`, seeds 1-3, from scratch (`--fresh`), with `RANKS` MPI ranks and the derived inputs of 02.
Every output is compared with `outputs/reference/reference_audit_sha256.csv`: merged outputs byte for byte, gzip
files by their decompressed bytes, part files as the sorted set of full-precision rows (so the number of ranks does
not matter); the signature folder names (a hash of every parameter and of the haul table) are part of the keys.

**Status: placeholder.** The fingerprint is generated with this cell (`MAKE_AUDIT_REFERENCE = True`) from the
derived inputs of the full audited run after the full run ends, and shipped in `outputs/reference/`. Until then the
step reports NOT YET AVAILABLE. It needs the prebuilt derived-input pack (a local rebuild is not guaranteed
byte-identical, see 02, section 0b).

**Expected result:** `REPRODUCTION CHECK: PASSED - exact match with the reference` (`mismatches: 0`).

In [ ]:
import time
ref_csv = os.path.join(REF_DIR, repro.AUDIT_REF)
do_run = MAKE_AUDIT_REFERENCE or RUN_SUBSET is True or str(RUN_SUBSET) == "True" or \
         (RUN_SUBSET == "auto" and os.path.isfile(ref_csv))
if not do_run:
    AUDIT["d rerun"] = "NOT YET AVAILABLE" if not os.path.isfile(ref_csv) else "SKIPPED (RUN_SUBSET = False)"
    print(f"(d) {AUDIT['d rerun']}: no fingerprint at {ref_csv}" if not os.path.isfile(ref_csv) else "(d) skipped")
else:
    if not os.path.isfile(os.path.join(WORK_DIR, "config.yaml")):
        repro.make_workdir(PACKAGE_DIR, WORK_DIR, cache="prebuilt")
    chk = os.path.join(RESULTS_DIR, "_audit_subset")
    t0 = time.time()
    repro.run_audit_subset(WORK_DIR, chk, ranks=RANKS, lock=LOCK)
    print(f"rerun: {time.time() - t0:.0f} s")
    if MAKE_AUDIT_REFERENCE:
        os.makedirs(REF_DIR, exist_ok=True)
        repro.write_fingerprint(repro.fingerprint(chk), ref_csv)
        AUDIT["d rerun"] = "REFERENCE WRITTEN"
        print("reference written:", ref_csv)
    elif os.path.isfile(ref_csv):
        ok = repro.compare_reference(chk, ref_csv)
        print("REPRODUCTION CHECK:", "PASSED - exact match with the reference" if ok else "FAILED")
        AUDIT["d rerun"] = "PASS" if ok else "FAIL"
    else:
        AUDIT["d rerun"] = "NOT YET AVAILABLE (ran, but no fingerprint to compare with)"
print("(d)", AUDIT["d rerun"])

## (e) Headline numbers from the shipped per-seed tables
Island years = (last finish day + 1) / 90 working days per season; island man-hours = the sum of `mh_total` over the
7,167 sites (rule of `code/tools/v12_summary.py`). The cell recomputes the mean, minimum and maximum over the 100 seeds
of `MOST_REALISTIC_AUDITED_LIT_P10` from `outputs/summary/per_seed_full_audited.csv` and compares them with
`outputs/summary/summary_full_audited.csv`; with `PARTS_DIR` set it also recomputes the per-seed table from the part
files (`seed_NNNN.sites.csv.gz`) of the results package.

**Status: placeholder** until the results package is public (the file names and columns are fixed in
`outputs/summary/README.md`).

**Expected result:** every row `equal = True`, and the island years and man-hours printed for the headline scenario.

In [ ]:
head = repro.headline_from_summaries(SUMMARY_DIR, parts_dir=PARTS_DIR)
if head is None:
    AUDIT["e headline"] = "NOT YET AVAILABLE"
else:
    display(head)
    AUDIT["e headline"] = "PASS" if head.equal.all() else "FAIL"
print("(e)", AUDIT["e headline"])

## Summary
**Expected result:** (a), (b), (c) PASS; (d) and (e) PASS once the fingerprint and the summary tables are shipped (NOT YET AVAILABLE before).

In [ ]:
for k, v in AUDIT.items():
    print(f"{k:12s} {v}")
assert not [k for k, v in AUDIT.items() if v == "FAIL"], "audit: at least one step FAILED (see above)"
